## ML based integration pipeline (Sorted Neighbourhood Blocker)

In [1]:
from pathlib import Path

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
from PyDI.io import load_parquet, load_csv

amazon_books = load_parquet(
    DATA_DIR / "amazon_new.parquet",
    name="amazon_books"
)

goodreads = load_parquet(
    DATA_DIR / "goodreads_new.parquet",
    name="goodreads"
)

metabooks = load_parquet(
  DATA_DIR / "metabooks_new.parquet",
  name="metabooks"
)

In [3]:
train_m2a = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="train_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2a = load_csv(
    MLDS_DIR / "test_MA.csv",
    name="test_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

train_m2g = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="train_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2g = load_csv(
    MLDS_DIR / "test_MG.csv",
    name="test_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

In [ ]:
from PyDI.entitymatching import StringComparator, NumericComparator

comparators_m2a = [
    StringComparator(column='title',similarity_function='jaccard'),
    StringComparator(column='title',similarity_function='cosine'),
    StringComparator(column='title',similarity_function='levenshtein'),
    StringComparator(column='title',similarity_function='jaro_winkler'),
    
    StringComparator(column='author',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='author',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='author',similarity_function='levenshtein', preprocess=str.lower),

    StringComparator(column='publisher',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='levenshtein', preprocess=str.lower),
    StringComparator(column='isbn_clean',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='isbn_clean',similarity_function='levenshtein', preprocess=str.lower),

    NumericComparator(column='publish_year',max_difference=1)
]

comparators_m2g = comparators_m2a + [
    StringComparator(column='genres', similarity_function='jaccard', preprocess=str.lower, list_strategy='concatenate'),
    StringComparator(column='genres', similarity_function='jaccard', preprocess=str.lower, list_strategy='best_match'),
    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
    NumericComparator(column="rating", max_difference=0.2)
]

/Users/abd/Developer/team_project_data_integration/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from PyDI.entitymatching import FeatureExtractor

feature_extractor_m2a = FeatureExtractor(comparators_m2a)
feature_extractor_m2g = FeatureExtractor(comparators_m2g)

train_features_m2a = feature_extractor_m2a.create_features(
    metabooks, amazon_books, train_m2a[['id1', 'id2']], labels=train_m2a['label'], id_column='id'
)

train_features_m2g = feature_extractor_m2g.create_features(
    metabooks, goodreads, train_m2g[['id1', 'id2']], labels=train_m2g['label'], id_column='id'
)

feature_columns_m2a = [col for col in train_features_m2a.columns if col not in ['id1', 'id2', 'label']]
X_train_m2a = train_features_m2a[feature_columns_m2a]
y_train_m2a = train_features_m2a['label']

feature_columns_m2g = [col for col in train_features_m2g.columns if col not in ['id1', 'id2', 'label']]
X_train_m2g = train_features_m2g[feature_columns_m2g]
y_train_m2g = train_features_m2g['label']

training_datasets = [(X_train_m2a, y_train_m2a),(X_train_m2g, y_train_m2g)]

In [6]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
import pandas as pd

# classifiers
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42)
}

# Define parameter grids
param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'class_weight': ['balanced', None],
        'min_samples_split': [2, 5]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    },
    'SVC': {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto']
    },
    'LogisticRegression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
    }
}


scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_models = []


for data in training_datasets:
    grid_search_results = {}
    best_overall_score = -1
    best_overall_model = None
    best_model_name = None

    for name, model in classifiers.items():
        print(f"Running GridSearchCV for {name}...")
        
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            scoring=scorer,
            cv=cv_folds,
            n_jobs=-1,
            verbose=0
        )
        
        grid.fit(data[0], data[1])
        
        grid_search_results[name] = {
            'grid_search': grid,
            'best_score': grid.best_score_,
            'best_params': grid.best_params_,
            'best_estimator': grid.best_estimator_
        }

        if grid.best_score_ > best_overall_score:
            best_overall_model = grid.best_estimator_

    best_models.append(best_overall_model)

Running GridSearchCV for RandomForestClassifier...
Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...
Running GridSearchCV for RandomForestClassifier...
Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...


In [ ]:
from PyDI.entitymatching import SortedNeighbourhoodBlocker
from PyDI.entitymatching import MLBasedMatcher


blocker_m2a = SortedNeighbourhoodBlocker(
    metabooks, amazon_books,
    key='title',
    window=20,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)


blocker_m2g = SortedNeighbourhoodBlocker(
    metabooks, goodreads,
    key='title',
    window=20,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

ml_matcher_m2a = MLBasedMatcher(feature_extractor_m2a)
ml_matcher_m2g = MLBasedMatcher(feature_extractor_m2g)

correspondences_m2a = ml_matcher_m2a.match(
    metabooks, amazon_books,
    candidates=blocker_m2a,
    id_column='id',
    trained_classifier=best_models[0]
)

correspondences_m2g = ml_matcher_m2g.match(
    metabooks, goodreads,
    candidates=blocker_m2g,
    id_column='id',
    trained_classifier=best_models[1]
)

In [8]:
# correspondences_m2a.to_csv(CORR_DIR / "ml_corr_m2a.csv", index=False)
# correspondences_m2g.to_csv(CORR_DIR / "ml_corr_m2g.csv", index=False)

In [9]:
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust
import pandas as pd

amazon_books.attrs["trust_score"] = 3
metabooks.attrs["trust_score"] = 2
goodreads.attrs["trust_score"] = 1

all_correspondences = pd.concat([correspondences_m2a, correspondences_m2g], ignore_index=True)

len(all_correspondences)

31814

In [ ]:
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title', longest_string)
strategy.add_attribute_fuser('author', longest_string)
strategy.add_attribute_fuser('publish_year', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publisher', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('isbn_clean', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('price', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('rating', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres', union)

In [11]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_ml_sn_blocker.jsonl")

fused_ml_snblocker = engine.run(
    datasets=[amazon_books, metabooks, goodreads],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)

print(f'Fused rows: {len(fused_ml_snblocker):,}')
display(fused_ml_snblocker.head())

Fused rows: 30,185


,_id,_fusion_group_id,_fusion_sources,price,genres,title_norm,rating,id,author,page_count,...,language,author_norm,title,rating_number,isbn_clean,publish_year,_fusion_confidence,_fusion_metadata,edition,book_format
0,amazon_92449,group_0,"[metabooks, amazon_books]",8.98,"[['Engineering', 'Transportation']]",transport phenomena a unified approach,4.7,amazon_92449,Robert S. Brodkey,847.0,...,English,robert s brodkey,Transport Phenomena: A Unified Approach,2,0070079633,1987.0,0.769231,"{'price_rule': 'prefer_higher_trust', 'price_s...",NaN,NaN
1,amazon_173439,group_1,"[metabooks, amazon_books]",9.41,"[['Romance', 'Historical']]",apache bride,4.0,amazon_173439,Joanne Redd,NaN,...,English,joanne redd,Apache Bride,3,0440205131,1990.0,0.692308,"{'price_rule': 'prefer_higher_trust', 'price_s...",NaN,NaN
2,metabooks_224528,group_2,"[metabooks, amazon_books]",8.23,"[['Teen', 'Young Adult', 'Literature', 'Fictio...",go ask alice,4.4,metabooks_224528,Simon & Schuster,192.0,...,English,simon schuster,Go Ask Alice,5855,0689817851,1998.0,0.782967,"{'genres_rule': 'union', 'genres_sources': ['m...",NaN,NaN
3,metabooks_84009,group_3,"[metabooks, amazon_books]",NaN,"[['Romance', 'Historical']]",i thee wed vanza,4.5,metabooks_84009,Amanda Quick,384.0,...,English,amanda quick,I Thee Wed (Vanza),959,0553574108,2000.0,0.706731,"{'genres_rule': 'union', 'genres_sources': ['m...",NaN,NaN
4,metabooks_8124,group_4,"[metabooks, amazon_books]",36.34,"[['Literature', 'Fiction', 'Genre Fiction']]",journey,4.3,metabooks_8124,James A. Michener,244.0,...,English,james a michener,Journey,1146,0394578260,1989.0,0.769231,"{'genres_rule': 'union', 'genres_sources': ['m...",NaN,NaN


In [12]:
fused_ml_snblocker.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30185 entries, 0 to 30184
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   _id                 30185 non-null  object 
 1   _fusion_group_id    30185 non-null  object 
 2   _fusion_sources     30185 non-null  object 
 3   price               27975 non-null  float64
 4   genres              28964 non-null  object 
 5   title_norm          30185 non-null  object 
 6   rating              30185 non-null  float64
 7   id                  30185 non-null  object 
 8   author              30185 non-null  object 
 9   page_count          27473 non-null  float64
 10  publisher           30185 non-null  object 
 11  language            30069 non-null  object 
 12  author_norm         30185 non-null  object 
 13  title               30185 non-null  object 
 14  rating_number       30185 non-null  int64  
 15  isbn_clean          30185 non-null  object 
 16  publ

In [13]:
fused_ml_snblocker.to_parquet(PIPELINE_DIR / "fused_ml_snblocker.parquet")